# 00. Inicjalizacja środowiska i przygotowanie zbioru testowego (Holdout)
### Instalacja wymaganych bibliotek
Instalujemy pakiety niezbędne do pobrania danych z Hugging Face, przetwarzania obrazów oraz ekstrakcji cech za pomocą PyTorch (`datasets`, `huggingface_hub`, `torch`, `torchvision`, `pillow`, `pyyaml`), a następnie restartujemy środowisko Pythona.

In [0]:
%pip install -U datasets huggingface_hub fsspec pillow pyyaml numpy torch torchvision
dbutils.library.restartPython()

### Konfiguracja Unity Catalog i tabeli stanu (`pipeline_state`)
Wczytujemy parametry z pliku `config.yaml`, tworzymy wirtualny dysk (Volume) na surowe pliki obrazów oraz inicjalizujemy tabelę Delta `pipeline_state`, która przechowuje stan potoku (numer ostatniej paczki oraz przesunięcie `offset` w zbiorze danych).

In [0]:
import yaml
import os

config_path = "config.yaml" if os.path.exists("config.yaml") else "../config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["storage"]["catalog_name"]
SCHEMA = config["storage"]["schema_name"]
VOLUME = config["storage"]["volume_name"]
TEST_SIZE = config["pipeline"]["holdout_test_size"]
HF_DATASET = config["dataset"]["hf_name"]

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_state (
    dataset_name STRING,
    last_batch_id INT,
    last_offset INT,
    updated_at TIMESTAMP
)
""")

existing = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.pipeline_state WHERE dataset_name = '{HF_DATASET}'").collect()
if len(existing) == 0:
    spark.sql(f"INSERT INTO {CATALOG}.{SCHEMA}.pipeline_state VALUES ('{HF_DATASET}', 0, 0, current_timestamp())")
    print(f"Zainicjalizowano licznik w pipeline_state dla zbioru: {HF_DATASET} (offset = 0)")
else:
    print(f"Tabela pipeline_state ma już wpis dla {HF_DATASET}:", existing[0].asDict())

### Przygotowanie ekstraktora cech (`MobileNetV3-Small`)
Ładujemy pre-trenowaną sieć konwolucyjną `MobileNetV3-Small`, zastępujemy jej klasyfikator warstwą `Identity` (dzięki czemu sieć zwraca wektor 576 cech na każdy obraz) i zamrażamy wagi. Definiujemy również standardowe przekształcenia obrazu (skalowanie do `224x224` oraz normalizację).

In [0]:
import torch
import torch.nn as nn
from torchvision import models, transforms

weights = models.MobileNet_V3_Small_Weights.DEFAULT
backbone = models.mobilenet_v3_small(weights=weights)

backbone.classifier = nn.Identity()

for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Ekstraktor cech MobileNetV3-Small gotowy (zwraca 576 cech na zdjęcie)!")

### Budowa zbalansowanego zbioru testowego (`silver_test_holdout`)
Pobieramy strumieniowo zdjęcia ze zbioru testowego tak, aby uzyskać idealnie zbalansowany podzbiór (po 50 zdjęć na każdą z 10 klas, łącznie 500 obrazów). Wyliczamy dla nich wektory cech i zapisujemy je w tabeli Delta `silver_test_holdout`, która posłuży jako stały zbiór ewaluacyjny dla wszystkich trenowanych modeli.

In [0]:
from datasets import load_dataset
from datetime import datetime
from collections import defaultdict

NUM_CLASSES = 10
SAMPLES_PER_CLASS = TEST_SIZE // NUM_CLASSES  

print(f"Pobieranie zbalansowanego zbioru testowego: po {SAMPLES_PER_CLASS} zdjęć dla każdej z {NUM_CLASSES} klas (łącznie {TEST_SIZE})...")

test_stream = load_dataset(
    HF_DATASET, 
    split=config["dataset"]["test_split"], 
    streaming=True
)

class_counts = defaultdict(int)
test_records = []
tensors_batch = []
meta_batch = []
accepted_count = 0

with torch.no_grad():
    for item in test_stream:
        label_id = int(item[config["dataset"]["label_col"]])
        
        if class_counts[label_id] >= SAMPLES_PER_CLASS:
            continue
            
        class_counts[label_id] += 1
        accepted_count += 1
        
        img = item[config["dataset"]["image_col"]].convert("RGB")
        raw_filename = item.get(config["dataset"]["id_col"], f"test_{accepted_count}.jpg")
        class_name = raw_filename.split("/")[0] if "/" in raw_filename else f"class_{label_id}"
        
        tensors_batch.append(preprocess(img))
        meta_batch.append((accepted_count - 1, raw_filename, label_id, class_name))
        
        if len(tensors_batch) == 64 or accepted_count == TEST_SIZE:
            batch_tensor = torch.stack(tensors_batch)
            embeddings = backbone(batch_tensor).numpy()
            
            for meta, emb in zip(meta_batch, embeddings):
                test_records.append({
                    "test_idx": meta[0],
                    "filename": meta[1],
                    "label_id": meta[2],
                    "class_name": meta[3],
                    "features": [float(x) for x in emb],
                    "created_at": datetime.utcnow()
                })
            tensors_batch, meta_batch = [], []
            
        if accepted_count == TEST_SIZE:
            break

test_df = spark.createDataFrame(test_records)
test_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_test_holdout")

print(f"Gotowe! Zapisano {len(test_records)} zbalansowanych rekordów w {CATALOG}.{SCHEMA}.silver_test_holdout.")

display(
    spark.sql(f"""
        SELECT label_id, class_name, COUNT(*) as test_samples
        FROM {CATALOG}.{SCHEMA}.silver_test_holdout
        GROUP BY label_id, class_name
        ORDER BY label_id
    """)
)